# Patient Profiles with OMOPy

This notebook demonstrates `omopy.profiles` — adding patient-level enrichment to `CdmTable` and `CohortTable` objects.

We cover:
- **Demographics** — age, sex, observation periods, date of birth
- **Categories** — binning continuous variables into groups
- **Cohort Intersect** — flag, count, date, and days relative to another cohort
- **Table Intersect** — flag, count, date, and days relative to a CDM table
- **Concept Intersect** — flag and count for concept sets
- **Death Information** — flag, date, days to death
- **Utility Functions** — cohort name, concept name, CDM name
- **Column Helpers** — resolving start/end date columns by table name

## 1. Configuration

In [1]:
DB_PATH = "../data/synthea.duckdb"
CDM_SCHEMA = "base"
WRITE_SCHEMA = "main"

## 2. Setup — Connect and Generate Cohort

Connect to the Synthea database (27 persons) and generate a concept-based cohort for two conditions.

In [2]:
from omopy.connector import cdm_from_con, generate_concept_cohort_set
from omopy.generics import Codelist

cdm = cdm_from_con(DB_PATH, cdm_schema=CDM_SCHEMA, write_schema=WRITE_SCHEMA)
print(cdm)

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)


In [3]:
concept_set = Codelist({"sinusitis": [40481087], "hypertension": [320128]})
cdm = generate_concept_cohort_set(cdm, concept_set, name="conditions")
cohort = cdm["conditions"]
print(cohort)
print(cohort.collect())

CohortTable('conditions', source='duckdb', cohorts=2)
shape: (9, 4)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date │
│ ---                  ┆ ---        ┆ ---               ┆ ---             │
│ i64                  ┆ i64        ┆ date              ┆ date            │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      │
│ 2                    ┆ 16         ┆ 1989-07-09        ┆ 2024-01-21      │
│ 1                 

## 3. Adding Demographics

### All at once with `add_demographics`

`add_demographics` enriches a table with age, sex, prior/future observation, and date of birth in a single call.

In [4]:
from omopy.profiles import (
    add_age,
    add_date_of_birth,
    add_demographics,
    add_future_observation,
    add_in_observation,
    add_prior_observation,
    add_sex,
)

enriched = add_demographics(cohort, cdm)
print(enriched.collect())

shape: (9, 8)
┌──────────────┬────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────┬────────┐
│ cohort_defin ┆ subject_id ┆ cohort_star ┆ cohort_end_ ┆ prior_obser ┆ future_obse ┆ age ┆ sex    │
│ ition_id     ┆ ---        ┆ t_date      ┆ date        ┆ vation      ┆ rvation     ┆ --- ┆ ---    │
│ ---          ┆ i64        ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ i64 ┆ str    │
│ i64          ┆            ┆ date        ┆ date        ┆ i64         ┆ i64         ┆     ┆        │
╞══════════════╪════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════╪════════╡
│ 1            ┆ 2          ┆ 2023-06-17  ┆ 2023-10-26  ┆ 13729       ┆ 131         ┆ 54  ┆ Male   │
│ 2            ┆ 5          ┆ 1991-12-22  ┆ 2023-10-29  ┆ 0           ┆ 11634       ┆ 18  ┆ Male   │
│ 2            ┆ 15         ┆ 2005-06-29  ┆ 2024-01-24  ┆ 6166        ┆ 6783        ┆ 18  ┆ Female │
│ 2            ┆ 16         ┆ 1989-07-09  ┆ 2024-01-21  ┆ 2597        ┆ 12614

You can toggle individual components on or off:

In [5]:
# Only age and sex, no observation or DOB
enriched_slim = add_demographics(
    cohort,
    cdm,
    age=True,
    sex=True,
    prior_observation=False,
    future_observation=False,
    date_of_birth=False,
)
print(enriched_slim.collect())

shape: (9, 6)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬─────┬────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ age ┆ sex    │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ --- ┆ ---    │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i64 ┆ str    │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═════╪════════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 54  ┆ Male   │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 18  ┆ Male   │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 18  ┆ Female │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ 51  ┆ Male   │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 18  ┆ Female │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ 

### Individual demographic functions

Each demographic attribute can also be added individually.

In [6]:
# Add age
with_age = add_age(cohort, cdm)
print(with_age.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬─────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ age │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ --- │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i64 │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 54  │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 18  │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 18  │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ 51  │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 18  │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ 18  │
│ 2                    ┆ 21         ┆ 1973-08-15        ┆ 2023-05-24      ┆ 18  │
│ 

In [7]:
# Add sex
with_sex = add_sex(cohort, cdm)
print(with_sex.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ sex    │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---    │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ str    │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪════════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ Male   │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ Male   │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ Female │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ Male   │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ Female │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ Male   │
│ 2                    ┆ 21         ┆ 1973-08-15   

In [8]:
# Add prior observation (days from observation_period_start_date to index)
with_prior = add_prior_observation(cohort, cdm)
print(with_prior.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬───────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ prior_observation │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---               │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i64               │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═══════════════════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 13729             │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 0                 │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 6166              │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ 14943             │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 6331              │
│ 2                    ┆ 1

In [9]:
# Add future observation (days from index to observation_period_end_date)
with_future = add_future_observation(cohort, cdm)
print(with_future.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬────────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ future_observation │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---                │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i64                │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪════════════════════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 131                │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 11634              │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 6783               │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ 268                │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 8534               │
│ 2             

In [10]:
# Add date of birth
with_dob = add_date_of_birth(cohort, cdm)
print(with_dob.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬───────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ date_of_birth │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---           │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ date          │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═══════════════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 1968-09-25    │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 1973-10-28    │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 1987-05-06    │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ 1971-05-16    │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 1982-06-19    │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-

In [11]:
# Add in-observation flag (boolean: is person in an observation period at index?)
with_in_obs = add_in_observation(cohort, cdm)
print(with_in_obs.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ in_observation │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---            │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i8             │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪════════════════╡
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 1              │
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ 1              │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ 1              │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 1              │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 1              │
│ 2                    ┆ 16         ┆ 1989-07-09        

### Age groups via `add_demographics`

Pass `age_group` to `add_demographics` or `add_age` to bin ages directly.

In [12]:
age_groups = {"age_group": {"0-17": (0, 17), "18-64": (18, 64), "65+": (65, 150)}}

with_age_group = add_demographics(cohort, cdm, age_group=age_groups)
print(with_age_group.collect())

shape: (9, 9)
┌─────────────┬────────────┬─────────────┬─────────────┬───┬────────────┬─────┬───────────┬────────┐
│ cohort_defi ┆ subject_id ┆ cohort_star ┆ cohort_end_ ┆ … ┆ future_obs ┆ age ┆ age_group ┆ sex    │
│ nition_id   ┆ ---        ┆ t_date      ┆ date        ┆   ┆ ervation   ┆ --- ┆ ---       ┆ ---    │
│ ---         ┆ i64        ┆ ---         ┆ ---         ┆   ┆ ---        ┆ i64 ┆ str       ┆ str    │
│ i64         ┆            ┆ date        ┆ date        ┆   ┆ i64        ┆     ┆           ┆        │
╞═════════════╪════════════╪═════════════╪═════════════╪═══╪════════════╪═════╪═══════════╪════════╡
│ 1           ┆ 2          ┆ 2023-06-17  ┆ 2023-10-26  ┆ … ┆ 131        ┆ 54  ┆ 18-64     ┆ Male   │
│ 2           ┆ 5          ┆ 1991-12-22  ┆ 2023-10-29  ┆ … ┆ 11634      ┆ 18  ┆ 18-64     ┆ Male   │
│ 2           ┆ 15         ┆ 2005-06-29  ┆ 2024-01-24  ┆ … ┆ 6783       ┆ 18  ┆ 18-64     ┆ Female │
│ 2           ┆ 16         ┆ 1989-07-09  ┆ 2024-01-21  ┆ … ┆ 12614      ┆ 18 

In [13]:
# Same via add_age
with_age_group2 = add_age(cohort, cdm, age_group=age_groups)
print(with_age_group2.collect())

shape: (9, 6)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬─────┬───────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ age ┆ age_group │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ --- ┆ ---       │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i64 ┆ str       │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═════╪═══════════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 54  ┆ 18-64     │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 18  ┆ 18-64     │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 18  ┆ 18-64     │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ 51  ┆ 18-64     │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 18  ┆ 18-64     │
│ 2                    ┆ 19         ┆ 2003-09-

## 4. Adding Categories

`add_categories` bins a numeric column into labelled groups. This works on any table that already has the variable.

In [14]:
from omopy.profiles import add_categories

# First add age, then categorise it
with_age = add_age(cohort, cdm)

categorised = add_categories(
    with_age,
    variable="age",
    categories={"age_group": {"0-17": (0, 17), "18-64": (18, 64), "65+": (65, 150)}},
)
print(categorised.collect())

shape: (9, 6)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬─────┬───────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ age ┆ age_group │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ --- ┆ ---       │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i64 ┆ str       │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═════╪═══════════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 54  ┆ 18-64     │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 18  ┆ 18-64     │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 18  ┆ 18-64     │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ 51  ┆ 18-64     │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 18  ┆ 18-64     │
│ 2                    ┆ 19         ┆ 2003-09-

## 5. Cohort Intersect

These functions compare a cohort against a target cohort table. We use our `"conditions"` cohort as both source and target for demonstration.

The `window` parameter is a tuple `(lower, upper)` defining the day range relative to `index_date`. Use `float('inf')` for unbounded.

In [15]:
from omopy.profiles import (
    add_cohort_intersect_count,
    add_cohort_intersect_date,
    add_cohort_intersect_days,
    add_cohort_intersect_flag,
)

In [16]:
# Flag: does each person have a record in the target cohort within the window?
with_flag = add_cohort_intersect_flag(
    cohort,
    "conditions",
    cdm,
    window=(0, float("inf")),
)
print(with_flag.collect())

shape: (9, 6)
┌─────────────────┬────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ cohort_definiti ┆ subject_id ┆ cohort_start_d ┆ cohort_end_dat ┆ sinusitis_0_to ┆ hypertension_0 │
│ on_id           ┆ ---        ┆ ate            ┆ e              ┆ _inf           ┆ _to_inf        │
│ ---             ┆ i64        ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│ i64             ┆            ┆ date           ┆ date           ┆ i8             ┆ i8             │
╞═════════════════╪════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ 1               ┆ 17         ┆ 2023-10-02     ┆ 2023-12-24     ┆ 1              ┆ 1              │
│ 1               ┆ 2          ┆ 2023-06-17     ┆ 2023-10-26     ┆ 1              ┆ 0              │
│ 2               ┆ 19         ┆ 2003-09-08     ┆ 2024-01-01     ┆ 0              ┆ 1              │
│ 2               ┆ 5          ┆ 1991-12-22     ┆ 2023-10-29     ┆ 0         

In [17]:
# Count: how many target records per person?
with_count = add_cohort_intersect_count(
    cohort,
    "conditions",
    cdm,
    window=(0, float("inf")),
)
print(with_count.collect())

shape: (9, 6)
┌─────────────────┬────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ cohort_definiti ┆ subject_id ┆ cohort_start_d ┆ cohort_end_dat ┆ sinusitis_0_to ┆ hypertension_0 │
│ on_id           ┆ ---        ┆ ate            ┆ e              ┆ _inf           ┆ _to_inf        │
│ ---             ┆ i64        ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│ i64             ┆            ┆ date           ┆ date           ┆ i64            ┆ i64            │
╞═════════════════╪════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ 1               ┆ 17         ┆ 2023-10-02     ┆ 2023-12-24     ┆ 1              ┆ 1              │
│ 1               ┆ 2          ┆ 2023-06-17     ┆ 2023-10-26     ┆ 1              ┆ 0              │
│ 2               ┆ 19         ┆ 2003-09-08     ┆ 2024-01-01     ┆ 0              ┆ 1              │
│ 2               ┆ 5          ┆ 1991-12-22     ┆ 2023-10-29     ┆ 0         

In [18]:
# Date: date of the first target record within the window
with_date = add_cohort_intersect_date(
    cohort,
    "conditions",
    cdm,
    window=(0, float("inf")),
    order="first",
)
print(with_date.collect())

shape: (9, 6)
┌─────────────────┬────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ cohort_definiti ┆ subject_id ┆ cohort_start_d ┆ cohort_end_dat ┆ sinusitis_0_to ┆ hypertension_0 │
│ on_id           ┆ ---        ┆ ate            ┆ e              ┆ _inf           ┆ _to_inf        │
│ ---             ┆ i64        ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│ i64             ┆            ┆ date           ┆ date           ┆ date           ┆ date           │
╞═════════════════╪════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ 1               ┆ 2          ┆ 2023-06-17     ┆ 2023-10-26     ┆ 2023-06-17     ┆ null           │
│ 2               ┆ 19         ┆ 2003-09-08     ┆ 2024-01-01     ┆ null           ┆ 2003-09-08     │
│ 2               ┆ 5          ┆ 1991-12-22     ┆ 2023-10-29     ┆ null           ┆ 1991-12-22     │
│ 2               ┆ 15         ┆ 2005-06-29     ┆ 2024-01-24     ┆ null      

In [19]:
# Days: days from index to the first target record
with_days = add_cohort_intersect_days(
    cohort,
    "conditions",
    cdm,
    window=(0, float("inf")),
    order="first",
)
print(with_days.collect())

shape: (9, 6)
┌─────────────────┬────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ cohort_definiti ┆ subject_id ┆ cohort_start_d ┆ cohort_end_dat ┆ sinusitis_0_to ┆ hypertension_0 │
│ on_id           ┆ ---        ┆ ate            ┆ e              ┆ _inf           ┆ _to_inf        │
│ ---             ┆ i64        ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│ i64             ┆            ┆ date           ┆ date           ┆ i64            ┆ i64            │
╞═════════════════╪════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ 1               ┆ 2          ┆ 2023-06-17     ┆ 2023-10-26     ┆ 0              ┆ null           │
│ 2               ┆ 19         ┆ 2003-09-08     ┆ 2024-01-01     ┆ null           ┆ 0              │
│ 2               ┆ 5          ┆ 1991-12-22     ┆ 2023-10-29     ┆ null           ┆ 0              │
│ 2               ┆ 15         ┆ 2005-06-29     ┆ 2024-01-24     ┆ null      

### Using windows and target cohort IDs

Restrict to events in the 365 days before the index date, targeting a specific cohort ID.

In [20]:
# Flag for hypertension records in the year before index
flag_prior_year = add_cohort_intersect_flag(
    cohort,
    "conditions",
    cdm,
    target_cohort_id=[2],
    window=(-365, -1),
    index_date="cohort_start_date",
)
print(flag_prior_year.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬────────────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ hypertension_m365_to_m │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ 1                      │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ ---                    │
│                      ┆            ┆                   ┆                 ┆ i8                     │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪════════════════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ 1                      │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 0                      │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      ┆ 1                      │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ 0

## 6. Table Intersect

Like cohort intersect, but targets a raw CDM table (e.g. `"drug_exposure"`, `"condition_occurrence"`).

In [21]:
from omopy.profiles import (
    add_table_intersect_count,
    add_table_intersect_date,
    add_table_intersect_days,
    add_table_intersect_flag,
)

In [22]:
# Flag: any drug_exposure record?
tbl_flag = add_table_intersect_flag(
    cohort,
    "drug_exposure",
    cdm,
    window=(0, float("inf")),
)
print(tbl_flag.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬────────────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ drug_exposure_0_to_inf │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---                    │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i8                     │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪════════════════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ 1                      │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ 1                      │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 1                      │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 1                      │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 1

In [23]:
# Count: number of drug_exposure records per person
tbl_count = add_table_intersect_count(
    cohort,
    "drug_exposure",
    cdm,
    window=(0, float("inf")),
)
print(tbl_count.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬────────────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ drug_exposure_0_to_inf │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---                    │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i64                    │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪════════════════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ 2                      │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ 75                     │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 70                     │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 97                     │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 2

In [24]:
# Date: first drug_exposure date within window
tbl_date = add_table_intersect_date(
    cohort,
    "drug_exposure",
    cdm,
    window=(0, float("inf")),
    order="first",
)
print(tbl_date.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬────────────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ drug_exposure_0_to_inf │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---                    │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ date                   │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪════════════════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ 2023-12-24             │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ 2003-09-08             │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 1991-12-22             │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 2005-06-29             │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 2

In [25]:
# Days: days from index to first drug_exposure
tbl_days = add_table_intersect_days(
    cohort,
    "drug_exposure",
    cdm,
    window=(0, float("inf")),
    order="first",
)
print(tbl_days.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬────────────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ drug_exposure_0_to_inf │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---                    │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i64                    │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪════════════════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ 83                     │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ 0                      │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 0                      │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 0                      │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 1

### Windowed table intersect

In [26]:
# Count condition_occurrence records in the 180 days after index
tbl_count_post = add_table_intersect_count(
    cohort,
    "condition_occurrence",
    cdm,
    window=(0, 180),
    index_date="cohort_start_date",
)
print(tbl_count_post.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬────────────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ condition_occurrence_0 │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ _to_180                │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ ---                    │
│                      ┆            ┆                   ┆                 ┆ i64                    │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪════════════════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ 1                      │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 2                      │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ 1                      │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 1

## 7. Concept Intersect

Intersect against specific concept IDs using a `Codelist`. This searches across relevant clinical tables.

In [27]:
from omopy.profiles import add_concept_intersect_count, add_concept_intersect_flag

In [28]:
# Define a concept set for the intersection
target_concepts = Codelist({"sinusitis": [40481087], "hypertension": [320128]})

# Flag: does the person have any record matching these concepts?
concept_flag = add_concept_intersect_flag(
    cohort,
    target_concepts,
    cdm,
    window=(0, float("inf")),
)
print(concept_flag.collect())

shape: (9, 6)
┌─────────────────┬────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ cohort_definiti ┆ subject_id ┆ cohort_start_d ┆ cohort_end_dat ┆ sinusitis_0_to ┆ hypertension_0 │
│ on_id           ┆ ---        ┆ ate            ┆ e              ┆ _inf           ┆ _to_inf        │
│ ---             ┆ i64        ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│ i64             ┆            ┆ date           ┆ date           ┆ i8             ┆ i8             │
╞═════════════════╪════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ 1               ┆ 2          ┆ 2023-06-17     ┆ 2023-10-26     ┆ 1              ┆ 0              │
│ 2               ┆ 19         ┆ 2003-09-08     ┆ 2024-01-01     ┆ 0              ┆ 1              │
│ 2               ┆ 5          ┆ 1991-12-22     ┆ 2023-10-29     ┆ 0              ┆ 1              │
│ 2               ┆ 15         ┆ 2005-06-29     ┆ 2024-01-24     ┆ 0         

In [29]:
# Count: how many records match?
concept_count = add_concept_intersect_count(
    cohort,
    target_concepts,
    cdm,
    window=(0, float("inf")),
)
print(concept_count.collect())

shape: (9, 6)
┌─────────────────┬────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ cohort_definiti ┆ subject_id ┆ cohort_start_d ┆ cohort_end_dat ┆ sinusitis_0_to ┆ hypertension_0 │
│ on_id           ┆ ---        ┆ ate            ┆ e              ┆ _inf           ┆ _to_inf        │
│ ---             ┆ i64        ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│ i64             ┆            ┆ date           ┆ date           ┆ i64            ┆ i64            │
╞═════════════════╪════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ 1               ┆ 2          ┆ 2023-06-17     ┆ 2023-10-26     ┆ 1              ┆ 0              │
│ 2               ┆ 19         ┆ 2003-09-08     ┆ 2024-01-01     ┆ 0              ┆ 1              │
│ 2               ┆ 5          ┆ 1991-12-22     ┆ 2023-10-29     ┆ 0              ┆ 1              │
│ 2               ┆ 15         ┆ 2005-06-29     ┆ 2024-01-24     ┆ 0         

In [30]:
# Windowed: concepts in the year before index
concept_flag_prior = add_concept_intersect_flag(
    cohort,
    target_concepts,
    cdm,
    window=(-365, -1),
    index_date="cohort_start_date",
)
print(concept_flag_prior.collect())

shape: (9, 6)
┌─────────────────┬────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ cohort_definiti ┆ subject_id ┆ cohort_start_d ┆ cohort_end_dat ┆ sinusitis_m365 ┆ hypertension_m │
│ on_id           ┆ ---        ┆ ate            ┆ e              ┆ _to_m1         ┆ 365_to_m1      │
│ ---             ┆ i64        ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│ i64             ┆            ┆ date           ┆ date           ┆ i8             ┆ i8             │
╞═════════════════╪════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ 1               ┆ 2          ┆ 2023-06-17     ┆ 2023-10-26     ┆ 0              ┆ 0              │
│ 2               ┆ 19         ┆ 2003-09-08     ┆ 2024-01-01     ┆ 0              ┆ 0              │
│ 2               ┆ 5          ┆ 1991-12-22     ┆ 2023-10-29     ┆ 0              ┆ 0              │
│ 2               ┆ 15         ┆ 2005-06-29     ┆ 2024-01-24     ┆ 0         

## 8. Death Information

Add death-related columns to the cohort.

In [31]:
from omopy.profiles import add_death_date, add_death_days, add_death_flag

In [32]:
# 0/1 flag for whether the person has a death record
with_death_flag = add_death_flag(cohort, cdm)
print(with_death_flag.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬───────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ death │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---   │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i8    │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═══════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ 0     │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ 0     │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ 0     │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ 0     │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ 0     │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 0     │
│ 2                    ┆ 16         ┆ 1989-07-09        ┆ 2024

In [33]:
# Date of death (null if alive)
with_death_date = add_death_date(cohort, cdm)
print(with_death_date.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬───────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ date_of_death │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---           │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ date          │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═══════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ null          │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ null          │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ null          │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ null          │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ null          │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-

In [34]:
# Days from index date to death
with_death_days = add_death_days(cohort, cdm, index_date="cohort_start_date")
print(with_death_days.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬───────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ days_to_death │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---           │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ i64           │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═══════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ null          │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ null          │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ null          │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ null          │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ null          │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-

## 9. Utility Functions

These add metadata columns that are useful for labelling and reporting.

In [35]:
from omopy.profiles import add_cdm_name, add_cohort_name, add_concept_name

In [36]:
# Add cohort name from the cohort settings table
with_name = add_cohort_name(cohort)
print(with_name.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬──────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ cohort_name  │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---          │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ str          │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪══════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ sinusitis    │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ sinusitis    │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ hypertension │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ hypertension │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ hypertension │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ 

In [37]:
# Add concept name — useful on clinical tables with concept_id columns
drug_exposure = cdm["drug_exposure"]
with_concept = add_concept_name(drug_exposure, cdm, column="drug_concept_id")
print(with_concept.collect())

shape: (663, 24)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ drug_expo ┆ person_id ┆ drug_conc ┆ drug_expo ┆ … ┆ drug_sour ┆ route_sou ┆ dose_unit ┆ drug_con │
│ sure_id   ┆ ---       ┆ ept_id    ┆ sure_star ┆   ┆ ce_concep ┆ rce_value ┆ _source_v ┆ cept_id_ │
│ ---       ┆ i64       ┆ ---       ┆ t_date    ┆   ┆ t_id      ┆ ---       ┆ alue      ┆ name     │
│ i64       ┆           ┆ i32       ┆ ---       ┆   ┆ ---       ┆ str       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆ date      ┆   ┆ i32       ┆           ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 661       ┆ 26        ┆ 40224805  ┆ 2017-12-2 ┆ … ┆ 40224805  ┆ null      ┆ null      ┆ 1 ML med │
│           ┆           ┆           ┆ 5         ┆   ┆           ┆           ┆           ┆ roxyprog │
│           ┆           ┆           ┆           ┆   ┆           ┆         

In [38]:
# Add the CDM database name as a column
with_cdm = add_cdm_name(cohort, cdm)
print(with_cdm.collect())

shape: (9, 5)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┬─────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date ┆ cdm_name    │
│ ---                  ┆ ---        ┆ ---               ┆ ---             ┆ ---         │
│ i64                  ┆ i64        ┆ date              ┆ date            ┆ str         │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╪═════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      ┆ dbt-synthea │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      ┆ dbt-synthea │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      ┆ dbt-synthea │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      ┆ dbt-synthea │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      ┆ dbt-synthea │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      ┆ dbt-synthe

## 10. Column Helpers

`start_date_column` and `end_date_column` resolve the standard date column name for a given OMOP table.

In [39]:
from omopy.profiles import end_date_column, start_date_column

# Each OMOP table has its own date column naming convention
for table in [
    "condition_occurrence",
    "drug_exposure",
    "observation_period",
    "visit_occurrence",
]:
    print(f"{table}: start={start_date_column(table)}, end={end_date_column(table)}")

condition_occurrence: start=condition_start_date, end=condition_end_date
drug_exposure: start=drug_exposure_start_date, end=drug_exposure_end_date
observation_period: start=observation_period_start_date, end=observation_period_end_date
visit_occurrence: start=visit_start_date, end=visit_end_date


## 11. Summary

The `omopy.profiles` module provides a composable set of functions for enriching `CdmTable` and `CohortTable` objects with patient-level information:

| Category | Functions |
|---|---|
| Demographics | `add_demographics`, `add_age`, `add_sex`, `add_prior_observation`, `add_future_observation`, `add_date_of_birth`, `add_in_observation` |
| Categories | `add_categories` |
| Cohort Intersect | `add_cohort_intersect_flag/count/date/days` |
| Table Intersect | `add_table_intersect_flag/count/date/days` |
| Concept Intersect | `add_concept_intersect_flag/count` |
| Death | `add_death_flag`, `add_death_date`, `add_death_days` |
| Utility | `add_cohort_name`, `add_concept_name`, `add_cdm_name` |
| Helpers | `start_date_column`, `end_date_column` |

**Key patterns:**
- Most functions take `x` (a table) as the first argument and `cdm` as the second (`add_cohort_name` takes only `x`).
- Intersect functions accept `window=(lower, upper)` with `float('inf')` for unbounded, and `index_date` parameters.
- Functions return enriched table objects — chain them together for multi-attribute profiles.

**Next steps:** See `04_cohort_characteristics.ipynb` for summarising and visualising these enriched profiles across cohorts.